In [1]:
import glob
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from pyproj import Transformer
import matplotlib.colors as mcolors
from shapely.geometry import Polygon, Point
import cartopy.feature as cfeature

In [ ]:
# Combined data
vars = ['am2_seaice_conc', 'cdr_seaice_conc', 'ICECON', 'sic']

# Allocate for dataset information
names, ds_all = [], []

for file_path in glob.glob('/glade/work/skygale/pbi-data/combined/*.nc'):
    name = file_path[38:-3]
    names.append(name)
    print(f'Processing {name}...')

    # Open dataset
    ds = xr.open_dataset(file_path, chunks={})

    # Get sea ice
    for varname in vars:
        if varname in ds.data_vars:
            ds = ds[varname]
            break

    # Standardize DataArray names
    rename_dict = {}
    if 'ni' in ds.dims:
        rename_dict['ni'] = 'x'
    if 'nj' in ds.dims:
        rename_dict['nj'] = 'y'
    ds = ds.rename(rename_dict)

    # Convert any fractions to percentage
    min_val, max_val = np.nanmin(ds), np.nanmax(ds)
    if max_val <= 1.05:
        print('  Fraction! --> Converting to percentage...')
        ds = ds * 100

    # Reorder dimensions
    existing_dims = [dim for dim in ['time', 'y', 'x'] if dim in ds.dims]
    ds = ds.transpose(*existing_dims)

    ds_all.append(ds)

Processing G10016_PROTO_25km_2025...
  Fraction! --> Converting to percentage...
Processing AMSR2_12.5km_2012_2025...


In [ ]:
def get_hudson_masks(ds):
    """
    Create masks for West and South Hudson Bay based on x/y or lat/lon bounds.

    Returns:
        mask_west, mask_south : xarray.DataArray (boolean)
    """

    # Define bounds (lon/lat)
    west_bounds = {'lon_min': -95, 'lon_max': -88, 'lat_min': 56, 'lat_max': 63}
    south_bounds = {'lon_min': -88, 'lon_max': -75, 'lat_min': 51, 'lat_max': 60}

    if 'lat' in ds.coords and 'lon' in ds.coords:
        print("Using lat/lon coordinates for masking")

        mask_west = ((ds.lon >= west_bounds['lon_min']) & (ds.lon <= west_bounds['lon_max']) &
                     (ds.lat >= west_bounds['lat_min']) & (ds.lat <= west_bounds['lat_max']))

        mask_south = ((ds.lon >= south_bounds['lon_min']) & (ds.lon <= south_bounds['lon_max']) &
                      (ds.lat >= south_bounds['lat_min']) & (ds.lat <= south_bounds['lat_max']))

    elif 'x' in ds.coords and 'y' in ds.coords:
        print("Using x/y coordinates for masking")

        # Set up transformer
        transformer = Transformer.from_crs("EPSG:4326", "EPSG:3413", always_xy=True)

        # Define West polygon in lon/lat
        west_poly_lonlat = [
            (west_bounds['lon_min'], west_bounds['lat_min']),
            (west_bounds['lon_max'], west_bounds['lat_min']),
            (west_bounds['lon_max'], west_bounds['lat_max']),
            (west_bounds['lon_min'], west_bounds['lat_max']),
            (west_bounds['lon_min'], west_bounds['lat_min'])
        ]

        # Define South polygon in lon/lat
        south_poly_lonlat = [
            (south_bounds['lon_min'], south_bounds['lat_min']),
            (south_bounds['lon_max'], south_bounds['lat_min']),
            (south_bounds['lon_max'], south_bounds['lat_max']),
            (south_bounds['lon_min'], south_bounds['lat_max']),
            (south_bounds['lon_min'], south_bounds['lat_min'])
        ]

        # Transform both polygons to projected x/y
        west_poly_proj = [transformer.transform(lon, lat) for lon, lat in west_poly_lonlat]
        south_poly_proj = [transformer.transform(lon, lat) for lon, lat in south_poly_lonlat]

        # Create shapely polygons
        west_polygon = Polygon(west_poly_proj)
        south_polygon = Polygon(south_poly_proj)

        # Create 2D meshgrid of x and y coordinates from dataset
        xx, yy = np.meshgrid(ds.x.values, ds.y.values)

        # Flatten meshgrid for efficient point-in-polygon testing
        points = np.vstack((xx.ravel(), yy.ravel())).T

        # Create masks by checking if points fall inside polygons
        mask_west_flat = np.array([west_polygon.contains(Point(x, y)) for x, y in points])
        mask_south_flat = np.array([south_polygon.contains(Point(x, y)) for x, y in points])

        # Reshape masks back to 2D (y, x)
        mask_west = xr.DataArray(mask_west_flat.reshape(xx.shape), dims=('y', 'x'), coords={'y': ds.y, 'x': ds.x})
        mask_south = xr.DataArray(mask_south_flat.reshape(xx.shape), dims=('y', 'x'), coords={'y': ds.y, 'x': ds.x})

    else:
        raise ValueError("Dataset does not have expected coordinate system (lat/lon or x/y)")

    return mask_west, mask_south


def draw_box(ax, bounds, edgecolor='red', linewidth=2):
    """
    Draw a rectangular box on a Cartopy map.

    Parameters:
    - ax: matplotlib axis with cartopy projection
    - bounds: dict with keys 'lon_min', 'lon_max', 'lat_min', 'lat_max'
    """
    # Corners of the box in lon/lat
    lons = [bounds['lon_min'], bounds['lon_max'], bounds['lon_max'], bounds['lon_min'], bounds['lon_min']]
    lats = [bounds['lat_min'], bounds['lat_min'], bounds['lat_max'], bounds['lat_max'], bounds['lat_min']]

    # Plot the box using PlateCarree projection (geodetic)
    ax.plot(lons, lats, transform=ccrs.PlateCarree(), color=edgecolor, linewidth=linewidth)

In [ ]:
ds = ds_all[-1]
west_mask, south_mask = get_hudson_masks(ds)

# Check if any points are masked in
print("Any West Hudson Bay:", west_mask.any().item())
print("Any South Hudson Bay:", south_mask.any().item())

In [ ]:
# Select specific time
ds_sel = ds.isel(time=225)

# Define projections
crs = ccrs.Stereographic(
    central_latitude=90,
    central_longitude=-45
)

crs_nsidc = ccrs.Stereographic(
    central_latitude=90,
    central_longitude=-45,
    true_scale_latitude=70,
    false_easting=0,
    false_northing=0
)

# Define levels (boundaries for the colors)
levels = np.linspace(0, 100, 11)
norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=plt.cm.Blues_r.N, clip=True)

# Create plot
fig, ax = plt.subplots(1, 1, figsize=(6, 6), subplot_kw={'projection': crs})

# Ocean
ax.add_feature(cfeature.OCEAN.with_scale('50m'), zorder=0, facecolor='#08306B')

# Sea ice concentration
im = ax.pcolormesh(
    ds.x.values,
    ds.y.values,
    (ds_sel.where(west_mask) * 100).values,
    cmap='Blues_r',
    norm=norm,
    transform=crs_nsidc,
    zorder=1
)

im = ax.pcolormesh(
    ds.x.values,
    ds.y.values,
    (ds_sel.where(south_mask) * 100).values,
    cmap='Blues_r',
    norm=norm,
    transform=crs_nsidc,
    zorder=1
)

# Land
ax.add_feature(cfeature.LAND.with_scale('50m'), zorder=2, facecolor='gray', edgecolor='black', linewidth=0.5)

# Coastlines
ax.coastlines(resolution='50m', zorder=3)

# Approximate extent around Hudson Bay in degrees
hudson_extent = [-87, -80, 49, 67]
ax.set_extent(hudson_extent, crs=ccrs.PlateCarree())

# Gridlines
ax.gridlines(draw_labels=False, linewidth=0.5, alpha=0.5, color='white')

# West and South Hudson Bay boxes
west_bounds = {'lon_min': -95, 'lon_max': -88, 'lat_min': 56, 'lat_max': 63}
south_bounds = {'lon_min': -88, 'lon_max': -75, 'lat_min': 51, 'lat_max': 59}

draw_box(ax, west_bounds, edgecolor='red', linewidth=2)
draw_box(ax, south_bounds, edgecolor='red', linewidth=2)

ax.text(-90, 60, "West", transform=ccrs.PlateCarree(), fontsize=12, color='red', ha='center', va='center', fontweight='bold')
ax.text(-85, 54, "South", transform=ccrs.PlateCarree(), fontsize=12, color='red', ha='center', va='center', fontweight='bold')

# Colorbar with percentage formatting
cbar = plt.colorbar(im, orientation='vertical', boundaries=levels, ticks=levels, pad=0.05, shrink=0.7)
cbar.set_label('Sea Ice Concentration (%)', fontsize=12)

# Title in same style
date_str = ds_sel["time"].dt.strftime("%d %b %Y").item()
ax.set_title(f'Sea Ice Concentration, {date_str}', fontsize=16, pad=20)

plt.tight_layout()
plt.show()